# 第7章 自然言語処理と生成AI ― サンプルコード
『AI実装検定 A級・S級 完全版』の第7章のリストを、本文と同じ順に実行できます。

**使い方**：上のメニュー［ランタイム］→［すべてのセルを実行］。CPU のランタイムで動きます（GPU は不要）。

**本書の動作確認環境**（Python 3.13.17）：NumPy 2.5.3、scikit-learn 1.9.1、PyTorch 2.14.1、NLTK 3.10.4、Transformers 5.20.2

In [ ]:
# 版をそろえたいとき（任意）：本書と同じ版を入れます。通常は Colab の標準版でそのまま動きます。
# !pip install -q numpy==2.5.3 scikit-learn==1.9.1 torch==2.14.1 nltk==3.10.4 transformers==5.20.2
import sys; print(sys.version)


## リスト7.1　空白で分ける・単語と記号を分ける・文字単位

In [ ]:
import re

text = "I don't like cats, but I love dogs!"
print(text.split())                                    # 空白で分けるだけ
print(re.findall(r"\w+|[^\w\s]", text))                # 単語と記号を分ける
print(list("dogs"))                                    # 文字単位

**本書に載っている出力**
```
['I', "don't", 'like', 'cats,', 'but', 'I', 'love', 'dogs!']
['I', 'don', "'", 't', 'like', 'cats', ',', 'but', 'I', 'love', 'dogs', '!']
['d', 'o', 'g', 's']
```

## リスト7.2　Porter のステミングと、辞書による見出し語化

In [ ]:
from nltk.stem import PorterStemmer

ps = PorterStemmer()
words = ["running", "studies", "happily", "caresses", "better", "ran"]
print([ps.stem(w) for w in words])                     # ルールで語尾を削る：語幹は辞書の語とは限らない

lemma = {"better": "good", "ran": "run", "studies": "study", "running": "run"}   # 見出し語化は辞書が必要
print([lemma.get(w, w) for w in words])

**本書に載っている出力**
```
['run', 'studi', 'happili', 'caress', 'better', 'ran']
['run', 'study', 'happily', 'caresses', 'good', 'run']
```

## リスト7.3　CountVectorizer と TfidfVectorizer、手計算との一致

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

docs = ["cat sat", "cat cat ran", "dog ran"]
cv = CountVectorizer()
X = cv.fit_transform(docs)
print(cv.get_feature_names_out().tolist())
print(X.toarray().tolist())                            # 単語の出現回数（Bag of Words）

tv = TfidfVectorizer()                                 # 標準設定：平滑化した idf と L2 正規化
T = tv.fit_transform(docs)
print(np.round(tv.idf_, 4).tolist())
print(np.round(T.toarray()[1], 4).tolist())            # 2 つ目の文書

n = 3
df = np.array([2, 1, 2, 1])                            # cat, dog, ran, sat
idf = np.log((1 + n) / (1 + df)) + 1                   # 平滑化 idf
v = np.array([2, 0, 1, 0]) * idf                       # 文書 2：cat が 2 回、ran が 1 回
print(np.round(idf, 4).tolist(), np.round(v / np.linalg.norm(v), 4).tolist())

**本書に載っている出力**
```
['cat', 'dog', 'ran', 'sat']
[[1, 0, 0, 1], [2, 0, 1, 0], [0, 1, 1, 0]]
[1.2877, 1.6931, 1.2877, 1.6931]
[0.8944, 0.0, 0.4472, 0.0]
[1.2877, 1.6931, 1.2877, 1.6931] [0.8944, 0.0, 0.4472, 0.0]
```

## リスト7.4　コサイン類似度・単語の類推・スキップグラムの組

In [ ]:
import numpy as np

E = {"king": [0.9, 0.8, 0.1, 0.0], "queen": [0.9, 0.1, 0.8, 0.0], "man": [0.1, 0.9, 0.1, 0.0],
     "woman": [0.1, 0.1, 0.9, 0.0], "apple": [0.0, 0.1, 0.1, 0.9]}   # 軸：王族らしさ・男性・女性・果物
E = {k: np.array(v) for k, v in E.items()}

def cos(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

print(round(cos(E["king"], E["queen"]), 3), round(cos(E["king"], E["apple"]), 3))
target = E["king"] - E["man"] + E["woman"]             # king - man + woman
print(target.round(2).tolist())
best = max((k for k in E if k not in ("king", "man", "woman")), key=lambda k: cos(target, E[k]))
print(best)

sentence = ["I", "love", "deep", "learning"]
w = 1                                                  # 窓幅 1 のスキップグラム：中心語から周りの語を予測
pairs = [(sentence[i], sentence[j]) for i in range(len(sentence))
         for j in range(max(0, i - w), min(len(sentence), i + w + 1)) if i != j]
print(pairs)

**本書に載っている出力**
```
0.664 0.082
[0.9, 0.0, 0.9, 0.0]
queen
[('I', 'love'), ('love', 'I'), ('love', 'deep'), ('deep', 'love'), ('deep', 'learning'), ('learning', 'deep')]
```

## リスト7.5　LSTM と Transformer のテキスト分類（おもちゃのデータ）

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
pos, neg = ["good", "great", "fine"], ["bad", "awful", "poor"]
vocab = {w: i + 1 for i, w in enumerate(pos + neg + ["movie", "film", "plot", "the"])}   # 0 はパディング
g = torch.Generator().manual_seed(0)

def make(n):
    xs, ys = [], []
    for _ in range(n):
        y = int(torch.rand(1, generator=g) < 0.5)
        k = torch.randint(0, 3, (1,), generator=g).item()
        w = (pos if y else neg)[k]
        sent = ["the", ["movie", "film", "plot"][torch.randint(0, 3, (1,), generator=g).item()], w]
        xs.append([vocab[t] for t in sent]); ys.append(y)
    return torch.tensor(xs), torch.tensor(ys)

class LSTMClf(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(len(vocab) + 1, 8, padding_idx=0)
        self.lstm = nn.LSTM(8, 16, batch_first=True)
        self.fc = nn.Linear(16, 2)
    def forward(self, x):
        _, (h, _) = self.lstm(self.emb(x))
        return self.fc(h[-1])

class TransClf(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(len(vocab) + 1, 16, padding_idx=0)
        self.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(16, 2, 32, dropout=0.0, batch_first=True), 1)
        self.fc = nn.Linear(16, 2)
    def forward(self, x):
        return self.fc(self.enc(self.emb(x)).mean(dim=1))   # 全トークンの平均

Xtr, ytr = make(200)
Xte, yte = make(100)
for Model in (LSTMClf, TransClf):
    m = Model()
    opt = torch.optim.Adam(m.parameters(), lr=0.01)
    for _ in range(60):
        opt.zero_grad()
        nn.functional.cross_entropy(m(Xtr), ytr).backward()
        opt.step()
    acc = (m(Xte).argmax(1) == yte).float().mean().item()
    print(Model.__name__, sum(p.numel() for p in m.parameters()), acc)

**本書に載っている出力**
```
LSTMClf 1786 1.0
TransClf 2434 1.0
```

## リスト7.6　BIO タグから固有表現を取り出し、F1 を計算

In [ ]:
tokens = ["Taro", "Yamada", "visited", "Tokyo", "Tower"]
gold = ["B-PER", "I-PER", "O", "B-LOC", "I-LOC"]
pred = ["B-PER", "I-PER", "O", "B-LOC", "O"]           # 最後の語を取りこぼした

def spans(tags):                                       # BIO タグから (種類, 開始, 終了) を取り出す
    out, start, kind = set(), None, None
    for i, t in enumerate(tags + ["O"]):
        if t == "O" or t.startswith("B-"):
            if kind is not None:
                out.add((kind, start, i))
            start, kind = (i, t[2:]) if t.startswith("B-") else (None, None)
    return out

g, p = spans(gold), spans(pred)
print(sorted(g))
print(sorted(p))
tp = len(g & p)
prec, rec = tp / len(p), tp / len(g)
print(tp, round(prec, 3), round(rec, 3), round(2 * prec * rec / (prec + rec), 3))   # 固有表現単位の F1
print(sum(a == b for a, b in zip(gold, pred)) / len(gold))                          # トークン正解率

**本書に載っている出力**
```
[('LOC', 3, 5), ('PER', 0, 2)]
[('LOC', 3, 4), ('PER', 0, 2)]
1 0.5 0.5 0.5
0.8
```

## リスト7.7　perplexity と、クリップした 1-gram 適合率

In [ ]:
import numpy as np
from collections import Counter

probs = [0.5, 0.25, 0.125, 0.5]                        # モデルが正解の各単語に与えた確率
ppl = np.exp(-np.mean(np.log(probs)))
print(round(ppl, 4))                                   # perplexity = exp(平均の負の対数尤度)

ref = "the cat is on the mat".split()
hyp = "the the the cat on mat".split()
c_hyp, c_ref = Counter(hyp), Counter(ref)
clipped = sum(min(c, c_ref[w]) for w, c in c_hyp.items())    # 参照にある回数でクリップ
print(clipped, len(hyp), round(clipped / len(hyp), 4))       # 1-gram の適合率

**本書に載っている出力**
```
3.3636
5 6 0.8333
```

## リスト7.8　WordPiece トークナイザと、小さな BERT（重みは乱数）

In [ ]:
import os, tempfile
import torch
from transformers import BertConfig, BertModel, BertTokenizer

d = tempfile.mkdtemp()
vocab = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]", "i", "love", "play", "##ing", "un", "##happy", "model", "##s"]
with open(os.path.join(d, "vocab.txt"), "w") as f:
    f.write("\n".join(vocab))
tok = BertTokenizer(os.path.join(d, "vocab.txt"))
print(tok.tokenize("I love playing unhappy models"))   # WordPiece：未知の語は部分語に分ける
enc = tok(["i love models", "i love"], padding=True, return_tensors="pt")
print(enc["input_ids"].tolist())
print(enc["attention_mask"].tolist())                  # 1 = 本物のトークン、0 = パディング
cfg = BertConfig(vocab_size=len(vocab), hidden_size=32, num_hidden_layers=2,
                 num_attention_heads=2, intermediate_size=64)
out = BertModel(cfg)(**enc).last_hidden_state
print(out.shape)                                       # (バッチ, トークン数, 隠れ次元)

**本書に載っている出力**
```
['i', 'love', 'play', '##ing', 'un', '##happy', 'model', '##s']
[[2, 5, 6, 11, 12, 3], [2, 5, 6, 3, 0, 0]]
[[1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 0, 0]]
torch.Size([2, 6, 32])
```

## リスト7.9　Hugging Face で学習済みモデルを使う（インターネット接続が必要）

（本書では未実行のリストです。環境によっては追加のインストールや通信が必要です。）

In [ ]:
from transformers import pipeline

clf = pipeline("sentiment-analysis")            # 学習済みモデルを自動でダウンロードして使う
print(clf("This book is easy to follow."))      # [{'label': 'POSITIVE', 'score': 0.99...}] のような形式

from transformers import AutoTokenizer, AutoModel
tok = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

## リスト7.10　温度・top-k・top-p

In [ ]:
import numpy as np

logits = np.array([2.0, 1.0, 0.0, -1.0])

def softmax(z, T=1.0):
    e = np.exp(z / T - (z / T).max())
    return e / e.sum()

print(softmax(logits).round(3).tolist())
print(softmax(logits, 0.5).round(3).tolist())          # 温度を下げる：尖る
print(softmax(logits, 2.0).round(3).tolist())          # 温度を上げる：平らになる

p = softmax(logits)
idx = np.argsort(p)[::-1]
k2 = np.zeros_like(p); k2[idx[:2]] = p[idx[:2]]
print((k2 / k2.sum()).round(3).tolist())               # top-k（k = 2）：上位 2 つだけ残して再正規化
cum = np.cumsum(p[idx])
nuc = idx[: int(np.searchsorted(cum, 0.9) + 1)]        # top-p（p = 0.9）：累積確率が 0.9 に届くまで
kp = np.zeros_like(p); kp[nuc] = p[nuc]
print(sorted(nuc.tolist()), (kp / kp.sum()).round(3).tolist())

**本書に載っている出力**
```
[0.644, 0.237, 0.087, 0.032]
[0.865, 0.117, 0.016, 0.002]
[0.455, 0.276, 0.167, 0.102]
[0.731, 0.269, 0.0, 0.0]
[0, 1, 2] [0.665, 0.245, 0.09, 0.0]
```

## リスト7.11　TF-IDF による検索と、プロンプトの組み立て

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

docs = ["The return policy allows refunds within 30 days.",
        "Shipping takes 3 to 5 business days.",
        "Our office is closed on Sundays."]
question = "How many days do I have to ask for a refund?"
tv = TfidfVectorizer(stop_words="english")
D = tv.fit_transform(docs)
q = tv.transform([question])
sims = (D @ q.T).toarray().ravel()                     # TF-IDF は L2 正規化済みなので内積 = コサイン類似度
print(sims.round(3).tolist(), int(sims.argmax()))     # refund と refunds が別の語なので、思った文書が選ばれない

from nltk.stem import PorterStemmer
ps = PorterStemmer()
stem_tv = TfidfVectorizer(stop_words="english", analyzer=lambda t: [ps.stem(w) for w in tv.build_analyzer()(t)])
D2 = stem_tv.fit_transform(docs)
sims = (D2 @ stem_tv.transform([question]).T).toarray().ravel()
print(sims.round(3).tolist(), int(sims.argmax()))     # 語幹にそろえると、正しい文書が選ばれる
context = docs[int(sims.argmax())]
prompt = f"Answer using only the context.\nContext: {context}\nQuestion: {question}"
print(prompt)

**本書に載っている出力**
```
[0.322, 0.402, 0.0] 1
[0.532, 0.243, 0.0] 0
Answer using only the context.
Context: The return policy allows refunds within 30 days.
Question: How many days do I have to ask for a refund?
```

## リスト7.12　VAE の KL と再パラメータ化、勾配の流れ

In [ ]:
import numpy as np
import torch

mu = np.array([0.5, -1.0])
logvar = np.array([0.0, np.log(4.0)])          # 分散 1 と 4
# KL( N(mu, s^2) || N(0, 1) ) = 0.5 * (mu^2 + s^2 - 1 - log s^2)
kl = 0.5 * (mu ** 2 + np.exp(logvar) - 1 - logvar)
print(kl.round(4).tolist(), round(float(kl.sum()), 4))

# 再パラメータ化：z = mu + sigma * eps（eps は固定の乱数）
eps = np.array([0.3, -0.5])
print((mu + np.exp(0.5 * logvar) * eps).tolist())

# eps に乱数を押し出すので、mu と logvar に勾配が流れる
m = torch.tensor([0.5, -1.0], requires_grad=True)
lv = torch.tensor([0.0, np.log(4.0)], dtype=torch.float32, requires_grad=True)
e = torch.tensor([0.3, -0.5])
z = m + torch.exp(0.5 * lv) * e
z.sum().backward()
print(m.grad.tolist(), [round(v, 4) for v in lv.grad.tolist()])

**本書に載っている出力**
```
[0.125, 1.3069] 1.4319
[0.8, -2.0]
[1.0, 1.0] [0.15, -0.5]
```

## リスト7.13　GAN の損失と、生成器の 2 つの式の勾配

In [ ]:
import numpy as np

# 判別器 D：本物なら 1、偽物なら 0 を出したい
d_real, d_fake = 0.9, 0.2                 # D(x)、D(G(z))
d_loss = -(np.log(d_real) + np.log(1 - d_fake))
print("D の損失", round(float(d_loss), 4))

# 生成器 G の損失：元の式と、実際によく使う式（non-saturating）
print("元の式", round(float(np.log(1 - d_fake)), 4))
print("よく使う式", round(float(-np.log(d_fake)), 4))

# D(G(z)) が 0 に近い（学習の初め）ときの勾配の大きさ
for d in (0.01, 0.5):
    g_orig = 1 / (1 - d)                  # d/dD log(1 - D) の絶対値
    g_ns = 1 / d                          # d/dD (-log D) の絶対値
    print(d, round(g_orig, 2), round(g_ns, 2))

# 最適な D と、そのときの目的関数（= -log 4）
p_data, p_g = 0.3, 0.3
d_star = p_data / (p_data + p_g)
print(d_star, round(float(-np.log(4)), 4))

**本書に載っている出力**
```
D の損失 0.3285
元の式 -0.2231
よく使う式 1.6094
0.01 1.01 100.0
0.5 2.0 2.0
0.5 -1.3863
```

## リスト7.14　拡散モデルの順方向：ᾱ の変化と、ノイズから元の値を復元する

In [ ]:
import numpy as np

T = 1000
beta = np.linspace(1e-4, 0.02, T)          # ノイズの量（線形スケジュール）
alpha = 1 - beta
abar = np.cumprod(alpha)                   # alpha の累積積
for t in (1, 100, 500, 1000):
    print(t, round(float(abar[t - 1]), 4))

# 順方向：x_t = sqrt(abar) * x0 + sqrt(1 - abar) * eps（一気に作れる）
x0, eps, t = 1.0, 0.5, 500
a = abar[t - 1]
xt = np.sqrt(a) * x0 + np.sqrt(1 - a) * eps
print(round(float(xt), 4))

# ノイズ eps を（完全に）当てられれば、x0 を復元できる
x0_hat = (xt - np.sqrt(1 - a) * eps) / np.sqrt(a)
print(round(float(x0_hat), 4))

**本書に載っている出力**
```
1 0.9999
100 0.897
500 0.0786
1000 0.0
0.7603
1.0
```